# Experiment 01 — From pixels to embeddings

**Question:** How does one 32 × 32 RGB image become 64 vectors, each containing 128 numbers?

We will follow one real CIFAR-10 image through every step. Nothing is trained in this experiment. Our goal is to understand the objects that a JEPA will later learn to use.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from jepa.data import CIFAR10_CLASSES, load_cifar10
from jepa.device import best_device
from jepa.patches import patchify

torch.manual_seed(7)
device = best_device()
device

## 1. Load one image

The label is shown to us so we know what we are looking at. It is not passed into the projection.

In [ ]:
dataset = load_cifar10(train=True)
image, label = dataset[0]

print('Label:', CIFAR10_CLASSES[label])
print('Shape:', tuple(image.shape))
print('Numbers in the image:', image.numel())

plt.imshow(image.permute(1, 2, 0))
plt.title(CIFAR10_CLASSES[label])
plt.axis('off');

## 2. Divide it into patches

A 4 × 4 RGB patch contains `4 × 4 × 3 = 48` values. An image that is 32 pixels wide and high contains `8 × 8 = 64` such patches.

In [ ]:
patches = patchify(image.unsqueeze(0), patch_size=4)
print('Input shape: ', tuple(image.unsqueeze(0).shape))
print('Patch shape: ', tuple(patches.shape))
print('First patch contains:', patches[0, 0].numel(), 'values')

The leading `1` is the batch size: we currently have one image. The result therefore has the shape `(1, 64, 48)`.

## 3. Project every patch into 128 dimensions

The same linear projection is applied to all 64 patches. Its parameters are random for now, so these vectors do not yet carry learned meaning.

In [ ]:
projection = nn.Linear(48, 128).to(device)
with torch.no_grad():
    embeddings = projection(patches.to(device)).cpu()

print('Patch shape:    ', tuple(patches.shape))
print('Embedding shape:', tuple(embeddings.shape))

In [ ]:
plt.figure(figsize=(11, 4))
plt.imshow(embeddings.squeeze(0), aspect='auto', cmap='coolwarm')
plt.colorbar(label='value')
plt.xlabel('Embedding dimension')
plt.ylabel('Patch number')
plt.title('64 untrained patch embeddings, each with 128 dimensions');

## What we learned

The image still begins with three RGB channels. Patchifying rearranges its 3,072 values into 64 groups of 48. The projection then maps every group to a 128-number vector.

We have created the shape of a representation, but not its meaning. Training in the next experiments will decide what information these vectors preserve.